# Разметка дыхания эксперимента 3

**Статус:** производитель кандидатов интервалов для последующей ручной
разметки. Порядок дыхательных режимов эксперимента 2 не используется.

Каждая запись РНЦХ имеет собственную последовательность режимов. Автоматический
поиск отмечает только продолжительные участки с малой локальной вариабельностью
и не присваивает им физиологические названия без протокола и ручного контроля.


## Входы и выход

Внешняя конфигурация задаёт обезличенный record_id, относительный путь CSV,
колонку дыхательного сигнала и, если она подтверждена первичным протоколом,
ожидаемую последовательность режимов. Постороннее не одновременное исследование
на другом приборе в конфигурацию не включается.

Выходной JSON хранит хеш CSV, оценённую частоту дискретизации, кандидаты
малоподвижных интервалов и статус pending_manual_review. Только человек
сопоставляет интервалы с командами протокола и устанавливает accepted.


In [ ]:
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CONFIG_ENV = "KALMYKOV_EXP03_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp03_paths.example.json"
    )
CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
RECORDINGS = {item["record_id"]: item for item in CONFIG["recordings"]}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_record(path):
    frame = pd.read_csv(path)
    frame.columns = [
        "time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm",
        "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm",
    ]
    return frame

def sampling_frequency(frame):
    time = frame["time_s"].to_numpy(dtype=float)
    delta = np.diff(time)
    if len(delta) == 0 or np.any(~np.isfinite(delta)) or np.any(delta <= 0):
        raise ValueError("time_s должен быть конечным и строго возрастающим")
    median_dt = float(np.median(delta))
    jitter_fraction = float(np.median(np.abs(delta - median_dt)) / median_dt)
    return 1.0 / median_dt, jitter_fraction


OUT_DIR = DERIVED_ROOT / "exp03" / "annotations" / "breathing"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALGORITHM_VERSION = "exp03-quiet-segments-v1"


In [ ]:
def quiet_segments(frame, column, win_s=1.0, min_duration_s=4.0):
    time = frame["time_s"].to_numpy(dtype=float)
    signal = frame[column].to_numpy(dtype=float)
    fs_hz, _ = sampling_frequency(frame)
    window = max(5, int(win_s * fs_hz))
    rolling_std = (
        pd.Series(signal)
        .rolling(window, center=True, min_periods=max(3, window // 2))
        .std()
        .to_numpy()
    )
    threshold = max(0.06, float(np.nanpercentile(rolling_std, 20) * 2.0))
    quiet = rolling_std < threshold
    intervals = []
    start = 0
    while start < len(quiet):
        if not quiet[start]:
            start += 1
            continue
        stop = start
        while stop < len(quiet) and quiet[stop]:
            stop += 1
        if stop > start and time[stop - 1] - time[start] >= min_duration_s:
            intervals.append([float(time[start]), float(time[stop - 1])])
        start = stop
    return intervals, threshold


In [ ]:
annotations = []
for record_id, spec in RECORDINGS.items():
    source_path = DATA_ROOT / spec["relative_path"]
    frame = read_record(source_path)
    fs_hz, jitter_fraction = sampling_frequency(frame)
    intervals, threshold = quiet_segments(
        frame, spec["respiration_column"]
    )
    input_sha256 = sha256_file(source_path)
    output = {
        "schema_version": 1,
        "annotation_type": "breathing",
        "algorithm_version": ALGORITHM_VERSION,
        "record_id": record_id,
        "input": {
            "relative_path": spec["relative_path"],
            "sha256": input_sha256,
            "sampling_frequency_hz": fs_hz,
            "sampling_frequency_source": "median_diff_time_s",
            "relative_time_step_jitter": jitter_fraction,
        },
        "respiration_column": spec["respiration_column"],
        "candidate_quiet_intervals_s": intervals,
        "protocol_mode_sequence": spec.get("mode_sequence", []),
        "heuristic": {
            "quiet_threshold_channel_units": threshold,
            "mode_labels_assigned_automatically": False,
        },
        "reviewed_modes": [],
        "qc": {
            "status": "pending_manual_review",
            "reviewer": None,
            "reviewed_at": None,
            "notes": None,
        },
    }
    (OUT_DIR / f"{record_id}.json").write_text(
        json.dumps(output, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    annotations.append(output)
    print(record_id, len(intervals), output["qc"]["status"])
print("Каталог:", OUT_DIR)


## Ручной контроль

Рецензент проверяет каждый интервал по форме сигнала, командам протокола и
состоянию подключения каналов. Ориентировочная цветная разметка старого
ноутбука 16 может служить только навигацией и не копируется как результат.


In [ ]:
CHECK_RECORD_ID = None
if CHECK_RECORD_ID is None:
    print("Задайте CHECK_RECORD_ID.")
else:
    annotation = json.loads(
        (OUT_DIR / f"{CHECK_RECORD_ID}.json").read_text(encoding="utf-8")
    )
    source_path = DATA_ROOT / annotation["input"]["relative_path"]
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("Хеш исходного CSV изменился")
    frame = read_record(source_path)
    time = frame["time_s"].to_numpy(dtype=float)
    column = annotation["respiration_column"]
    fig, axis = plt.subplots(figsize=(14, 4))
    axis.plot(time, frame[column], linewidth=0.7)
    for start, stop in annotation["candidate_quiet_intervals_s"]:
        axis.axvspan(start, stop, alpha=0.2)
    axis.set_xlabel("Время, с")
    axis.set_ylabel(column)
    axis.set_title(
        f"{CHECK_RECORD_ID}: кандидаты интервалов; "
        f"QC={annotation['qc']['status']}"
    )
    plt.tight_layout()
    plt.show()
